# Importing Libraries

In [13]:
## Import modules
import os, sys
import numpy as np
import geopandas as gpd
import cftime
import gc
import shapely
import json
from ipywidgets import interact, FloatSlider
import ipywidgets as widgets
import matplotlib.pyplot as plt
# Import Plotly for interactive plotting
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as pc
from ipywidgets import interact, IntSlider, Dropdown, VBox, HBox
import ipywidgets as widgets

# Add the directory containing 'cmct' to the Python path
# Navigate two levels up to reach main CmCt dir
cmct_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir))

# Initialising Logger
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Import utilities for this comparison
sys.path.insert(0, cmct_dir)
from cmct.time_utils import *
from cmct.calving import *
from cmct.calving_modules.interpolation import *
from cmct.calving_modules.residual_calculation import *
# from cmct.calving_modules.json_to_netcdf import *
# from cmct.shapefile_utils import *

# Force initial garbage collection
gc.collect()

1299

In [14]:
# Reload modules to pick up any changes to imports
import importlib
import cmct.calving
import cmct.calving_modules.residual_calculation
from cmct.calving_modules.plotting_utils import *
from cmct.calving import calculate_basin_statistics, format_basin_stats
from cmct.calving import calculate_basin_statistics
importlib.reload(cmct.calving)
importlib.reload(cmct.calving_modules.residual_calculation)


# Re-import to ensure functions are available
from cmct.calving import *

# CONFIGURATION

In [15]:
# Observation Dataset
# Ice sheet
loc = "GIS"  # 'GIS' or 'AIS'

# Set the observation data dir path
obs_filename = cmct_dir + "/data/calving/observed_icemask_ismip_annual.nc"

# To use aggregation functions for basin
basin_aggregation = True  # IMPORTANT

basin_filename = cmct_dir + "/bin/Calving/GRE_Basins_IMBIE2_v1.3/GRE_Basins_IMBIE2_v1.3.shp"

# Set the Model Data dir path
# model_filename = cmct_dir + "/test/calving/ensemble/sftgif_B001_hist.nc"
model_filename = cmct_dir + "/test/calving/sftgif_GIS_JPL_ISSM_historical.nc"

# Set time range for comparison
start_year = 2007
end_year = 2010

# List of basins (ex ["NW", "NE"]) to compare if all -> "all", if none -> False
# If you do not know which basins are in the model, you can put "auto"
# NOTE: Align this list with the basins in the model.
basin_list = ["NW"]

# Output filetype and filename
filetype = "netcdf"  # netcdf or json or None
filename = "calving_comparison"

# Optional Configurations
interpolation_method = "slinear"  # 'nearest', 'linear', 'cubic'
accuracy_calculation_method = "mean"  # 'mean', 'RMS',

colors = {
    "CW": "blue",
    "NE": "red",
    "SE": "green",
    "SW": "orange",
    "NO": "purple",
    "NW": "brown",
}


# Loading all data files

In [16]:
# Check if observation file exist
if not os.path.exists(obs_filename):
    raise FileNotFoundError(f"Observation file not found: {obs_filename}")

# # Check if model file exist
if not os.path.exists(model_filename):
    raise FileNotFoundError(f"Model file not found: {model_filename}")



if basin_aggregation and not os.path.exists(basin_filename):
    raise FileNotFoundError(f"Basin shapefile not found: {basin_filename}")
    # Load basin shapes

print(basin_filename)
basins, basin_list = load_basins(basin_filename, basin_list)

print(obs_filename)
gsfc = load_gsfc_calving(obs_filename, basins)

print(model_filename)
model_res = load_model_calving(model_filename)

/Users/aditya_pachpande/Documents/GitHub/CmCt/bin/Calving/GRE_Basins_IMBIE2_v1.3/GRE_Basins_IMBIE2_v1.3.shp
/Users/aditya_pachpande/Documents/GitHub/CmCt/data/calving/observed_icemask_ismip_annual.nc
/Users/aditya_pachpande/Documents/GitHub/CmCt/test/calving/sftgif_GIS_JPL_ISSM_historical.nc


## Handelling Time Consistency

In [17]:
# Simplifying date data type
gsfc.ds["time"] = standardising_time_var(gsfc.time)
model_res.ds["time"] = standardising_time_var(model_res.time)

# Handelling Time Range
checking_calving_daterange(gsfc.time.values, model_res.time.values, start_year, end_year)


The selected dates 2007 to 2010 are within the overlapping data range.


# Interpolation

In [18]:
interpolater = Interpolater(model_res, gsfc)
model_res.ds = interpolater.interpolate()

2025-07-17 09:31:39,175 - INFO - Input x coordinates: [-720000. -715000. -710000. -705000. -700000. -695000. -690000. -685000.
 -680000. -675000. -670000. -665000. -660000. -655000. -650000. -645000.
 -640000. -635000. -630000. -625000. -620000. -615000. -610000. -605000.
 -600000. -595000. -590000. -585000. -580000. -575000. -570000. -565000.
 -560000. -555000. -550000. -545000. -540000. -535000. -530000. -525000.
 -520000. -515000. -510000. -505000. -500000. -495000. -490000. -485000.
 -480000. -475000. -470000. -465000. -460000. -455000. -450000. -445000.
 -440000. -435000. -430000. -425000. -420000. -415000. -410000. -405000.
 -400000. -395000. -390000. -385000. -380000. -375000. -370000. -365000.
 -360000. -355000. -350000. -345000. -340000. -335000. -330000. -325000.
 -320000. -315000. -310000. -305000. -300000. -295000. -290000. -285000.
 -280000. -275000. -270000. -265000. -260000. -255000. -250000. -245000.
 -240000. -235000. -230000. -225000. -220000. -215000. -210000. -20500

# Comparison and Residual Calculation 

In [19]:
print(type(basins))
years = np.arange(start_year, end_year + 1)

residuals = create_calving_dataset(gsfc, model_res, years, basins)

2025-07-17 09:31:47,142 - INFO - Starting optimized calving dataset creation...
2025-07-17 09:31:47,155 - INFO - Transformed basin NW: 4986 points
2025-07-17 09:31:47,157 - INFO - Grid dimensions: 2880 x 1680


<class 'dict'>


2025-07-17 09:31:47,413 - INFO - Computing residuals...
2025-07-17 09:31:47,441 - INFO - Computing statistics...
2025-07-17 09:31:47,473 - INFO - Creating basin assignments...
2025-07-17 09:31:47,474 - INFO - Data coordinates: X=[-719500.0, 959500.0], Y=[-3449500.0, -570500.0]
2025-07-17 09:31:47,475 - INFO - Basin polygon coordinates: X=[-607915.8, 245896.7], Y=[-1990296.2, -1151988.9]
2025-07-17 09:31:52,750 - INFO - Basin assignment complete. Unique basin IDs: [-1  0]
2025-07-17 09:31:52,753 - INFO -   Unassigned points: 4567530
2025-07-17 09:31:52,757 - INFO -   Basin 0 (NW): 270870 points
2025-07-17 09:31:52,760 - INFO - Basin assignment rate: 270870/4838400 (5.6%)
2025-07-17 09:31:52,769 - INFO - Creating xarray dataset...
2025-07-17 09:31:52,770 - INFO - model_data_all.shape: (4, 2880, 1680)
2025-07-17 09:31:52,770 - INFO - gsfc_data_all.shape: (4, 2880, 1680)
2025-07-17 09:31:52,770 - INFO - residuals_all.shape: (4, 2880, 1680)
2025-07-17 09:31:52,770 - INFO - basin_mask.shape:

In [20]:
residuals = load_residuals(residuals)


In [21]:
vars(residuals)


{'ds': <xarray.Dataset> Size: 310MB
 Dimensions:                 (time: 4, y: 2880, x: 1680, basin_id: 1)
 Coordinates:
   * time                    (time) int64 32B 2007 2008 2009 2010
   * x                       (x) float32 7kB -7.195e+05 -7.185e+05 ... 9.595e+05
   * y                       (y) float32 12kB -3.45e+06 -3.448e+06 ... -5.705e+05
     basin_names             (basin_id) <U2 8B 'NW'
 Dimensions without coordinates: basin_id
 Data variables:
     residual                (time, y, x) float32 77MB 0.0 0.0 0.0 ... 0.0 0.0
     basin                   (time, y, x) int32 77MB -1 -1 -1 -1 ... -1 -1 -1 -1
     gsfc_ice_mask           (time, y, x) float32 77MB 0.0 0.0 0.0 ... 0.0 0.0
     model_ice_mask          (time, y, x) float32 77MB 0.0 0.0 0.0 ... 0.0 0.0
     stats_avg_abs_residual  (time) float64 32B 0.02462 0.0246 0.0246 0.02462
     stats_rms_residual      (time) float64 32B 0.1319 0.1318 0.1319 0.1319
     stats_sum_residual      (time) float64 32B 5.066e+04 5.04e+04 .

# Statistics Calculation

In [25]:
basin_stats = calculate_basin_statistics(residuals)
print(format_basin_stats(basin_stats))

2025-07-17 09:32:12,474 - INFO - Starting basin statistics calculation
2025-07-17 09:32:12,475 - INFO - Processing 4 time steps and 1 basins
2025-07-17 09:32:12,547 - INFO - Basin statistics calculation completed


=== Statistics for Year 2007 ===
Basin | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
NW    |   270870 | -0.00082656 | -0.00079170 |  0.00008652 |   0.060868 |   0.060874
-------------------------------------------------------------------------------------


=== Statistics for Year 2008 ===
Basin | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
NW    |   270870 | -0.00096604 | -0.00095555 |  0.00008393 |   0.061331 |   0.061339
-------------------------------------------------------------------------------------


=== Statistics for Year 2009 ===
Basin | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
NW    |   270870 | -0.00109909 | -0.00108973 |  0.00008039 |   

# Plot Generation

## Configuration
- residuals are required for plot generation

In [23]:
# Which year do you want plots for?
year = 2007

cmap = "ocean"  # Leave default to Ocean :)
aspect = "auto"

# Plot config
plt.figure(figsize=(10, 6))
# plt.cm.viridis.set_bad('lightblue')  # Set color for NaN values


<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

In [24]:
create_interactive_residual_plot(residuals, year, basin_id=None)
    
create_basin_statistics_plot(basin_stats, year)
  
# Create interactive widgets
year_slider = IntSlider(
    value=start_year,
    min=start_year,
    max=end_year,
    step=1,
    description="Year:",
    continuous_update=False
)

basin_options = [('All Basins', -1)] + [(name, idx) for idx, name in enumerate(residuals.ds.basin_names.values)]
basin_dropdown = Dropdown(
    options=basin_options,
    value=-1,
    description="Basin:",
)

plot_type_dropdown = Dropdown(
    options=[('Residual Map', 'residual'), ('Basin Statistics', 'stats')],
    value='residual',
    description="Plot Type:"
)

# Calculate basin statistics for the interactive plots

interactive_plot(basin_stats, year, basin_id, plot_type)
    
# Create the interactive widget
interact(interactive_plot, 
         year=year_slider, 
         basin_id=basin_dropdown, 
         plot_type=plot_type_dropdown)

# Also create a combined view function
create_combined_dashboard(basin_stats, year)
        
# Combined dashboard
year_slider_combined = IntSlider(
    value=start_year,
    min=start_year,
    max=end_year,
    step=1,
    description="Year:",
    continuous_update=False
)

interact(create_combined_dashboard, year=year_slider_combined)

NameError: name 'basin_id' is not defined

In [ ]:
def create_time_series_plot(basin_stats, statistic='mean'):
    """
    Create an interactive time series plot showing how statistics change over time
    
    Parameters:
    -----------
    basin_stats : dict
        Basin statistics dictionary
    statistic : str
        Which statistic to plot ('mean', 'std', 'rms', 'count', etc.)
    """
    
    # Prepare data
    years = sorted(basin_stats.keys())
    basin_names = list(basin_stats[years[0]].keys())
    
    fig = go.Figure()
    
    # Add a trace for each basin
    for basin_name in basin_names:
        values = []
        for year in years:
            if basin_name in basin_stats[year] and basin_stats[year][basin_name]['count'] > 0:
                values.append(basin_stats[year][basin_name][statistic])
            else:
                values.append(None)
        
        fig.add_trace(go.Scatter(
            x=years,
            y=values,
            mode='lines+markers',
            name=basin_name,
            line=dict(width=2),
            marker=dict(size=8),
            hovertemplate=f'Year: %{{x}}<br>Basin: {basin_name}<br>{statistic.title()}: %{{y:.6f}}<extra></extra>'
        ))
    
    fig.update_layout(
        title=f'Time Series of {statistic.title()} by Basin',
        xaxis_title='Year',
        yaxis_title=statistic.title(),
        hovermode='x unified',
        width=900,
        height=500,
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=1.01
        )
    )
    
    return fig

def create_correlation_matrix(basin_stats, year):
    """
    Create a correlation matrix heatmap for different statistics
    """
    if year not in basin_stats:
        return None
    
    # Prepare data for correlation
    stats_data = []
    basin_names = []
    
    for basin_name, stats in basin_stats[year].items():
        if stats['count'] > 0:
            basin_names.append(basin_name)
            stats_data.append([
                stats['mean'],
                stats['std'],
                stats['rms'],
                stats['winsorized_mean'],
                stats['outlier_weighted_mean']
            ])
    
    if len(stats_data) < 2:
        return None
    
    import pandas as pd
    df = pd.DataFrame(stats_data, 
                     index=basin_names,
                     columns=['Mean', 'Std Dev', 'RMS', 'Winsorized Mean', 'Outlier Weighted Mean'])
    
    # Calculate correlation matrix
    corr_matrix = df.corr()
    
    # Create heatmap
    fig = go.Figure(data=go.Heatmap(
        z=corr_matrix.values,
        x=corr_matrix.columns,
        y=corr_matrix.columns,
        colorscale='RdBu',
        zmid=0,
        text=corr_matrix.values,
        texttemplate='%{text:.3f}',
        textfont={"size": 12},
        hoverongaps=False,
        hovertemplate='%{x} vs %{y}<br>Correlation: %{z:.3f}<extra></extra>'
    ))
    
    fig.update_layout(
        title=f'Statistics Correlation Matrix for Year {year}',
        width=600,
        height=600
    )
    
    return fig

# Create interactive widgets for time series
statistic_dropdown = Dropdown(
    options=[('Mean', 'mean'), ('Standard Deviation', 'std'), ('RMS', 'rms'), 
             ('Winsorized Mean', 'winsorized_mean'), ('Outlier Weighted Mean', 'outlier_weighted_mean')],
    value='mean',
    description="Statistic:"
)

def interactive_time_series(statistic):
    """Interactive time series plotting"""
    fig = create_time_series_plot(basin_stats, statistic)
    if fig:
        fig.show()

def interactive_correlation(year):
    """Interactive correlation matrix"""
    fig = create_correlation_matrix(basin_stats, year)
    if fig:
        fig.show()
    else:
        print(f"Cannot create correlation matrix for year {year}")

print("\n=== Additional Interactive Plots ===")
print("Time Series Analysis:")
interact(interactive_time_series, statistic=statistic_dropdown)

print("\nCorrelation Analysis:")
year_slider_corr = IntSlider(
    value=start_year,
    min=start_year,
    max=end_year,
    step=1,
    description="Year:",
    continuous_update=False
)
interact(interactive_correlation, year=year_slider_corr)


=== Additional Interactive Plots ===
Time Series Analysis:


interactive(children=(Dropdown(description='Statistic:', options=(('Mean', 'mean'), ('Standard Deviation', 'st…


Correlation Analysis:


interactive(children=(IntSlider(value=2007, continuous_update=False, description='Year:', max=2010, min=2007),…

<function __main__.interactive_correlation(year)>

# Calculating Basin Statistics